In [1]:
# Cell 1 — Imports
import json
import numpy as np
import pandas as pd
from collections import defaultdict
import re
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded!")

✅ Libraries loaded!


In [ ]:
# Cell 2 -- Final Roman Urdu dictionary (198 words, regression-fixed)
# History: started at 30 words -> expanded to 179 -> expansion process
# accidentally DROPPED 20 original words (kya, news, today, film, etc.)
# -> fixed 2026-08-08: restored all words, deduplicated, fixed 2 corrupted
# keys ("hogа" with Cyrillic char, "huا" malformed -> "hua").

roman_to_urdu_dict = {
    "aaj": "آج", "aao": "آؤ", "ab": "اب",
    "acha": "اچھا", "actor": "اداکار", "adalat": "عدالت",
    "afridi": "آفریدی", "aj": "آج", "ana": "آنا",
    "army": "فوج", "atta": "آٹا", "aur": "اور",
    "awam": "عوام", "aya": "آیا", "azam": "اعظم",
    "babar": "بابر", "bada": "بڑا", "bank": "بینک",
    "barish": "بارش", "batting": "بیٹنگ", "bayan": "بیان",
    "bhi": "بھی", "bijli": "بجلی", "bilawal": "بلاول",
    "bol": "بول", "bowling": "باؤلنگ", "bura": "برا",
    "business": "کاروبار", "century": "سنچری", "chawal": "چاول",
    "chota": "چھوٹا", "computer": "کمپیوٹر", "court": "عدالت",
    "cpec": "سی پیک", "cricket": "کرکٹ", "dena": "دینا",
    "dhoop": "دھوپ", "din": "دن", "diya": "دیا",
    "do": "دو", "dollar": "ڈالر", "doodh": "دودھ",
    "dost": "دوست", "drama": "ڈرامہ", "duniya": "دنیا",
    "dushman": "دشمن", "economy": "معیشت", "election": "انتخابات",
    "faisalabad": "فیصل آباد", "fakhar": "فخر", "fazal": "فضل",
    "fielding": "فیلڈنگ", "film": "فلم", "flood": "سیلاب",
    "football": "فٹبال", "four": "چوکا", "game": "گیم",
    "garmi": "گرمی", "gaya": "گیا", "geo": "جیو",
    "geya": "گیا", "gham": "غم", "ghar": "گھر",
    "ghee": "گھی", "goal": "گول", "government": "حکومت",
    "haar": "ہار", "hai": "ہے", "hain": "ہیں",
    "haris": "حارث", "hi": "ہی", "ho": "ہو",
    "hoga": "ہوگا", "hospital": "ہسپتال", "hui": "ہوئی",
    "hukumat": "حکومت", "huا": "ہوا", "imf": "آئی ایم ایف",
    "important": "اہم", "imran": "عمران", "india": "انڈیا",
    "internet": "انٹرنیٹ", "islamabad": "اسلام آباد", "jana": "جانا",
    "jao": "جاؤ", "jeet": "جیت", "ka": "کا",
    "kaisa": "کیسا", "kal": "کل", "kam": "کم",
    "karachi": "کراچی", "karna": "کرنا", "karo": "کرو",
    "ke": "کے", "khan": "خان", "khel": "کھیل",
    "khelna": "کھیلنا", "khush": "خوش", "khushi": "خوشی",
    "ki": "کی", "kiya": "کیا", "ko": "کو",
    "kuch": "کچھ", "kya": "کیا", "lahore": "لاہور",
    "leader": "رہنما", "lekin": "لیکن", "lena": "لینا",
    "liya": "لیا", "lo": "لو", "loss": "شکست",
    "madad": "مدد", "manga": "مانگا", "mangay": "مانگے",
    "market": "مارکیٹ", "masail": "مسائل", "masjid": "مسجد",
    "masla": "مسئلہ", "mat": "مت", "match": "میچ",
    "maut": "موت", "mehngai": "مہنگائی", "mein": "میں",
    "mobile": "موبائل", "mosam": "موسم", "muashi": "معاشی",
    "mulk": "ملک", "multan": "ملتان", "na": "نہ",
    "nahi": "نہیں", "naseem": "نسیم", "nateeja": "نتیجہ",
    "navy": "بحریہ", "nawaz": "نواز", "naya": "نیا",
    "ne": "نے", "news": "خبر", "ny": "نے",
    "out": "آؤٹ", "over": "اوور", "pakistan": "پاکستان",
    "pani": "پانی", "par": "پر", "pareshan": "پریشان",
    "party": "پارٹی", "peshawar": "پشاور", "petrol": "پیٹرول",
    "phir": "پھر", "pm": "وزیراعظم", "police": "پولیس",
    "price": "قیمت", "psl": "پی ایس ایل", "ptv": "پی ٹی وی",
    "purana": "پرانا", "qaum": "قوم", "quetta": "کوئٹہ",
    "raat": "رات", "raha": "رہا", "rawalpindi": "راولپنڈی",
    "rizwan": "رضوان", "roti": "روٹی", "rozgar": "روزگار",
    "run": "رن", "sab": "سب", "sadr": "صدر",
    "sardi": "سردی", "school": "اسکول", "score": "اسکور",
    "se": "سے", "sehat": "صحت", "serious": "سنگین",
    "shahbaz": "شہباز", "shaheen": "شاہین", "sharif": "شریف",
    "shikast": "شکست", "sirf": "صرف", "six": "چھکا",
    "siyasi": "سیاسی", "speech": "تقریر", "sy": "سے",
    "tabahi": "تباہی", "taleem": "تعلیم", "team": "ٹیم",
    "technology": "ٹیکنالوجی", "tha": "تھا", "thand": "ٹھنڈ",
    "thi": "تھی", "thy": "تھے", "today": "آج",
    "udas": "اداس", "vote": "ووٹ", "wazir": "وزیر",
    "wicket": "وکٹ", "win": "جیت", "zabardast": "زبردست",
    "zardari": "زرداری", "zindagi": "زندگی", "zyada": "زیادہ",
}

import difflib

def transliterate_roman_urdu(text, fuzzy_cutoff=0.75):
    """Dictionary lookup with fuzzy-match fallback for spelling variants."""
    dict_keys = list(roman_to_urdu_dict.keys())
    words = text.lower().split()
    translated_words = []
    for word in words:
        if word in roman_to_urdu_dict:
            translated_words.append(roman_to_urdu_dict[word])
            continue
        if len(word) >= 3:
            matches = difflib.get_close_matches(
                word, dict_keys, n=1, cutoff=fuzzy_cutoff
            )
            if matches:
                translated_words.append(roman_to_urdu_dict[matches[0]])
                continue
        translated_words.append(word)
    return ' '.join(translated_words)

print(f"✅ Final dictionary loaded!")
print(f"Total words: {len(roman_to_urdu_dict)}")
print(f"\nSample entries:")
for k, v in list(roman_to_urdu_dict.items())[:5]:
    print(f"  {k} → {v}")

# Save the corrected dictionary to disk so the rest of the
# pipeline (05_roman_urdu.ipynb, evaluation notebooks) uses it
import os
os.makedirs("../models", exist_ok=True)
with open("../models/roman_urdu_dict_expanded.json", "w", encoding="utf-8") as f:
    json.dump(roman_to_urdu_dict, f, ensure_ascii=False, indent=2)
print("\n✅ Saved to ../models/roman_urdu_dict_expanded.json")
